# Pyspark SQL Hints
------------------------
These are the:
Instructions you give Spark’s optimizer to influence how a query should execute.

Spark normally decides execution plans automatically using the Catalyst Optimizer and AQE, but hints let you suggest:

join strategies
partitioning behaviour
shuffle handling

Why SQL Hints Are Used

Sometimes Spark chooses a bad plan.

Example:

wrong join strategy
too many shuffles
skewed partitions
inefficient repartitioning

Hints help guide Spark toward a better execution plan.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In Spark SQL / Databricks SQL, a view is like a temporary virtual table created from a query.

Instead of saving data permanently, a view stores:

the SQL logic/query

You can then query it like a normal table.

In [0]:
epl_df = spark.read.table('sparkoptimization.tables.epl_final')
print(epl_df.count())
stadium_df = spark.read.table('sparkoptimization.tables.epl_stadiums')

1. Temporary View (temp view)

A temporary view exists:

only for the CURRENT Spark session
only for the CURRENT notebook/job
disappears when session ends

2. Global Temporary View (global temp view)

A global temp view is:

shared across notebooks/sessions
available to all users on the same cluster
stored in a special database called:


In [0]:
### There are two types of views temps views and global temp views both are temporary, global cvan be used in other spark session.

epl_df.createOrReplaceTempView('epl')
stadium_df.createOrReplaceTempView('stadiums')

# Join Using SQL
--------------------

In [0]:
# %sql
# SELECT * FROM epl e
# LEFT JOIN stadiums s
# ON e.HomeTeam = s.Team

''' THe above is the same as '''

df = spark.sql('''
SELECT * FROM epl e
LEFT JOIN stadiums s
ON e.HomeTeam = s.Team
''')
display(df)

#Adding SQL Hints
- Add the braodcase hint

| Hint                   | Purpose                                 |
| ---------------------- | --------------------------------------- |
| `broadcast`            | Force broadcast join                    |
| `merge`                | Prefer sort merge join                  |
| `shuffle_hash`         | Prefer shuffle hash join                |
| `shuffle_replicate_nl` | Replicate small table to all partitions |
| `repartition`          | Control partition count                 |
| `coalesce`             | Reduce partitions                       |
| `skew`                 | Help with skewed joins                  |


In [0]:
### The advice /hints can be rejected as well
df_opt = spark.sql("""
SELECT /*+ BROADCAST(s) */ *
FROM epl e
LEFT JOIN stadiums s
ON e.HomeTeam = s.Team
""")

display(df_opt)

In [0]:
### See what was applied to the table 
### We can see it did happen as we have  PhotonBroadcastHashJoin and BuildRight present.

'''Spark decided to:

broadcast the RIGHT table
which is stadiums s
to all executors
instead of shuffling both tables.
'''
df_opt.explain(True)